Soma total de vendas por dia , ordens com status de 'Delivered' e estate = NY


In [0]:
# Definir pastas do projetos em variaveis para facilitar
bronze_path   = '/Volumes/bikestore/default/bikestore/bronze/'
silver_path   = '/Volumes/bikestore/default/bikestore/silver/'
gold_path     = '/Volumes/bikestore/default/bikestore/gold/'
resource_path = '/Volumes/bikestore/default/bikestore/resource/origem/'

In [0]:
import pyspark.sql.functions as F

#df1 = spark.read.table("bikestore.logistics.silver_customers")
df2 = spark.read.table("bikestore.logistics.silver_orders")
#df3 = spark.read.table("bikestore.logistics.silver_product")

In [0]:
df_customers_gold = df2.select('shipped_date','total_sale','state','status')\
    .filter((F.col('status') == "Delivered") & (F.col('state') == "NY"))\
    .filter((F.col("shipped_date").isNotNull()))\
    .groupBy('shipped_date')\
    .agg(F.round(F.sum('total_sale'),2).alias('total_sale'))

In [0]:
#salvar arquivo parquet na silver como delta
df_customers_gold.write\
.format('delta')\
.mode('overwrite')\
.option("mergeSchema", "true")\
.save(gold_path+'sales_ny')


In [0]:
#criando tabela
df = df_customers_gold.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("bikestore.logistics.gold_sales_ny")